# BEE 4750 Homework 5: Mixed Integer and Stochastic Programming

**Name**:

**ID**:

> **Due Date**
>
> Thursday, 12/04/24, 9:00pm

## Overview

### Instructions

-   In Problem 1, you will use mixed integer programming to solve a
    waste load allocation problem.
-   In Problem 2, you will formulate a stochastic optimization problem.

### Load Environment

The following code loads the environment and makes sure all needed
packages are installed. This should be at the start of most Julia
scripts.

In [ ]:
import Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

  Activating project at `c:\Users\AllyK\bee 4750\hw5-ally,elliot\hw5-ally-elliot-1`
   Installed Measures ─────────── v0.3.3
   Installed GR_jll ───────────── v0.73.18+0
   Installed HiGHS_jll ────────── v1.12.0+0
   Installed OpenBLAS32_jll ───── v0.3.29+0
   Installed PlotUtils ────────── v1.4.4
   Installed OpenSSL ──────────── v1.6.0
   Installed MutableArithmetics ─ v1.6.7
   Installed Pango_jll ────────── v1.57.0+0
   Installed FFMPEG ───────────── v0.4.5
   Installed StaticArraysCore ─── v1.4.4
   Installed DataStructures ───── v0.19.3
   Installed JSON ─────────────── v1.3.0
   Installed GraphRecipes ─────── v0.5.15
   Installed METIS_jll ────────── v5.1.3+0
   Installed StatsBase ────────── v0.34.8
   Installed Adapt ────────────── v4.4.0
   Installed ChainRulesCore ───── v1.26.0
   Installed HiGHS ────────────── v1.20.1
   Installed StableRNGs ───────── v1.0.4
   Installed FFMPEG_jll ───────── v8.0.0+0
   Installed Interpolations ───── v0.16.2
   Installed JuMP ──────────────

In [2]:
using JuMP
using HiGHS
using DataFrames
using GraphRecipes
using Plots
using Measures
using MarkdownTables

## Problems (Total: 30 Points)

### Problem 1 (24 points)

Three cities are developing a coordinated municipal solid waste (MSW)
disposal plan. Three disposal alternatives are being considered: a
landfill (LF), a materials recycling facility (MRF), and a
waste-to-energy facility (WTE). The capacities of these facilities and
the fees for operation and disposal are provided below.

-   **LF**: Capacity 200 Mg, fixed cost \$2000/day, tipping cost
    \$50/Mg;
-   **MRF**: Capacity 350 Mg, fixed cost \$1500/day, tipping cost
    \$7/Mg, recycling cost \$40/Mg recycled;
-   **WTE**: Capacity 210 Mg, fixed cost \$2500/day, tipping cost
    \$60/Mg;

The MRF recycling rate is 40%, and the ash fraction of non-recycled
waste is 16% and of recycled waste is 14%. Transportation costs are
\$1.5/Mg-km, and the relative distances between the cities and
facilities are provided in the table below.

| **City/Facility** | **Landfill (km)** | **MRF (km)** | **WTE (km)** |
|:-----------------:|:-----------------:|:------------:|:------------:|
|         1         |         5         |      30      |      15      |
|         2         |        15         |      25      |      10      |
|         3         |        13         |      45      |      20      |
|        LF         |        \-         |      32      |      18      |
|        MRF        |        32         |      \-      |      15      |
|        WTE        |        18         |      15      |      \-      |

The fixed costs associated with the disposal options are incurred only
if the particular disposal option is implemented. The three cities
produce 100, 90, and 120 Mg/day of solid waste, respectively, with the
composition provided in the table below.

| **Component** | **% of total mass** | **Combustion ash** (%) | **MRF Recycling rate** (%) |
|:---------------------:|:--------------:|:---------------:|:---------------:|
| Food Wastes | 15 | 8 | 0 |
| Paper & Cardboard | 40 | 7 | 55 |
| Plastics | 5 | 5 | 15 |
| Textiles | 3 | 10 | 10 |
| Rubber, Leather | 2 | 15 | 0 |
| Wood | 5 | 2 | 30 |
| Yard Wastes | 18 | 2 | 40 |
| Glass | 4 | 100 | 60 |
| Ferrous | 2 | 100 | 75 |
| Aluminum | 2 | 100 | 80 |
| Other Metal | 1 | 100 | 50 |
| Miscellaneous | 3 | 70 | 0 |

The information in the above table will help you determine the overall
recycling and ash fractions. Note that the recycling residuals, which
may be sent to either landfill or the WTE, have different ash content
than the ash content of the original MSW. You will need to determine
these fractions to construct your mass balance constraints.

**Reminder**: Use `round(x; digits=n)` to report values to the
appropriate precision!

#### Problem 1.1

Based on the information above, calculate the overall recycling and ash
fractions for the waste produced by each city.

#### Problem 1.2

What are the decision variables for your optimization problem? Provide
notation and variable meaning.

#### Problem 1.3

Formulate the objective function. Make sure to include any needed
derivations or justifications for your equation(s).

#### Problem 1.4

Derive all relevant constraints. Make sure to include any needed
justifications or derivations.

#### Problem 1.5

Find the optimal solution (using `JuMP` to solve the problem). Report
the optimal objective value.

In [ ]:
meow = Model(HiGHS.Optimizer)
# facilities in order: WTE, MRF, LF
S = [120, 90, 100]          # S_i  = waste production by city i (Mg/day)
K = [210, 350, 200]         # K_j  = capacity of facility j (Mg)
f = [0.1641, 1-0.3775]      # f_k  = fraction of waste at facility k left as residue (unitless)
a = 1.5                     # a_ij = transportation cost from city i to facility j ($/Mg/km)
b = [60, 7 + 40 * f[2], 50] # b_j  = variable cost of facility j ($/Mg)
c = [2500, 1500, 2000]      # c_j  = fixed cost of facility j ($/day)
l1 = [15 30 5               # l1_ij = distance from city i to facility j (km)
      10 25 15
      20 45 13]
l2 = [0 32 18               # l2_kj = distance from facility k to facility j (km)
      32 0 15
      18 15 0]
@variable(meow, W[1:3,1:3] >= 0)
@variable(meow, R[1:2,1:3] >= 0)
fix(R[1,1], 0, force=true)
fix(R[1,2], 0, force=true)
fix(R[2,2], 0, force=true)
@variable(meow, Y[1:3], Bin)
@constraint(meow, cities_dispose_all[i=1:3], sum(W[i,j] for j=1:3) == S[i])
@constraint(meow, facility_capacity[j=1:3], sum(W[i,j] for i=1:3) + sum(R[k,j] for k=1:2) <= K[j])
@constraint(meow, facility_status[j=1:3], !Y[j] => {sum(W[i,j] for i=1:3) == 0})
@constraint(meow, residual13, R[1,3] == f[1] * sum(W[i,1] for i=1:3))
@constraint(meow, residual21_23, R[2,1] + R[2,3] == f[2] * sum(W[i,2] for i=1:3))
@objective(meow, Min, sum(a * l1[i,j] * W[i,j] for i=1:3, j=1:3) + sum(a * l2[k,j] * R[k,j] for k=1:2, j=1:3) + sum(c[j] * Y[j] + sum(b[i] * W[i,j] for i=1:3) for j=1:3))
optimize!(meow)

Running HiGHS 1.12.0 (git hash: 755a8e027): Copyright (c) 2025 HiGHS under MIT licence terms


#### Problem 1.6

Draw a diagram showing the flows of waste between the cities and the
facilities. Which facilities (if any) will not be used? Does this
solution make sense?

### Problem 2 (6 points)

Consider a two-period economic dispatch problem, based on the
multi-period example from Lecture 14 (on 10/29). The generator data,
including ramping constraints for each generator, is provided in
\`data/generators.csv.’ In period 1, the demand is
$d_1 = 1100 \text{MW}$. In period 2, the demand is projected to be
$d_2 = 1200 \text{MW}$, but there is a 25% probability that it is \$1500
. In the first period, the solar capacity factor is $0.9$ and the wind
capacity factor is $0.45$, but in the second period, there is some
uncertainty: the forecasted solar and wind capacity factors are $0.95$
and $0.4$, respectively, but there is a 30% probability that they are
$0.75$ and $0.5$. Your goal is to identify how to dispatch your
generators to minimize the cost of meeting demand.

#### Problem 2.1

Draw a scenario tree for this problem.

#### Problem 2.2

Formulate a stochastic linear program for this problem based on your
scenario tree from Problem 2.1 and the data in `data/generators.csv`.

## References

List any external references consulted, including classmates.